# Kanglish Review & Support Triage — Baselines, QLoRA Fine-Tuning & Eval Harness

Run this notebook on **Google Colab (T4 GPU)** or **Kaggle (P100/T4 GPU)**.

### What you'll learn & produce:
1. **Verify the Evaluation Harness** before any training.
2. **Baseline 1 (Zero-Shot)** & **Baseline 2 (Few-Shot)** on `Qwen/Qwen2.5-1.5B-Instruct`.
3. **4-bit QLoRA Fine-Tuning** with `trl.SFTTrainer` + `peft` (tracking train vs. validation loss).
4. **Post-Training Evaluation & Comparison Table** on the held-out `test.jsonl` split.
5. **Error Analysis** on remaining failure cases.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

## 1. Load Project Files & Verify Eval Harness
Upload your `kanglish-triage` folder (or clone it from your GitHub repo) into `/content/kanglish-triage` (Colab) or `/kaggle/working/kanglish-triage` (Kaggle), then set `PROJECT_DIR` below.

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path("/content/kanglish-triage")  # Update if using Kaggle or Drive
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "eval"))

!python -m unittest eval/test_evaluate.py -v

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load_jsonl, evaluate_predictions, print_report
from schema import SYSTEM_PROMPT

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

train_data = load_jsonl(Path("data/train.jsonl"))
val_data = load_jsonl(Path("data/val.jsonl"))
test_data = load_jsonl(Path("data/test.jsonl"))
print(f"Loaded splits -> train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}")

## 2. Shared Inference Helper
Using the exact same inference function for Zero-Shot, Few-Shot, and Fine-Tuned models ensures a fair benchmark.

In [ ]:
FEW_SHOT_EXAMPLES = [
    (
        "Indiranagar cafe ambience sakath agide, but filter coffee rate tumba jasti aytu guru.",
        '{"sentiment": "mixed", "category": "ambience", "urgency": "low", "english_summary": "Customer liked the Indiranagar cafe ambience but felt the filter coffee was overpriced."}'
    ),
    (
        "Order madi 1 hour aytu, delivery boy phone pick madta illa. Tumba late aytu.",
        '{"sentiment": "negative", "category": "delivery_service", "urgency": "high", "english_summary": "Customer\'s order has been delayed for an hour and the delivery partner is not answering calls."}'
    ),
    (
        "Hubballi khanavali jolada rotti mattu ennegayi bhari aiti, full maja bantu!",
        '{"sentiment": "positive", "category": "food_quality", "urgency": "low", "english_summary": "Customer loved the jolada rotti and ennegayi at the Hubballi restaurant."}'
    ),
]

def build_messages(text: str, few_shot: bool = False):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if few_shot:
        for ex_in, ex_out in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": ex_in})
            messages.append({"role": "assistant", "content": ex_out})
    messages.append({"role": "user", "content": text})
    return messages

@torch.inference_mode()
def run_eval_split(model, tokenizer, dataset, out_path: Path, few_shot: bool = False, max_new_tokens: int = 160):
    model.eval()
    preds = []
    for i, item in enumerate(dataset):
        messages = build_messages(item["text"], few_shot=few_shot)
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        preds.append({"id": item["id"], "raw_output": raw_output})
        if (i + 1) % 25 == 0:
            print(f"  Processed {i + 1}/{len(dataset)}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    report = evaluate_predictions(dataset, preds)
    print_report(report, max_failures=5)
    return report

## 3. Baseline Benchmarks (Zero-Shot & Few-Shot)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
print("--- Running Baseline 1: Zero-Shot ---")
zeroshot_report = run_eval_split(base_model, tokenizer, test_data, Path("eval/preds_zeroshot.jsonl"), few_shot=False)

print("\n--- Running Baseline 2: Few-Shot (3 examples) ---")
fewshot_report = run_eval_split(base_model, tokenizer, test_data, Path("eval/preds_fewshot.jsonl"), few_shot=True)

## 4. QLoRA Fine-Tuning (`trl.SFTTrainer` + `peft`)

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

def to_sft_record(item):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["text"]},
        {"role": "assistant", "content": json.dumps(item["label"], ensure_ascii=False)},
    ]
    return {"messages": messages}

train_ds = Dataset.from_list([to_sft_record(x) for x in train_data])
val_ds = Dataset.from_list([to_sft_record(x) for x in val_data])

model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="outputs/qwen2.5-1.5b-kanglish-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    load_best_model_at_end=True,
    fp16=True,
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model("outputs/best-lora-adapter")

## 5. Post-Training Evaluation & Comparison Table

In [ ]:
print("--- Running Fine-Tuned Model on Held-Out Test Set ---")
ft_report = run_eval_split(trainer.model, tokenizer, test_data, Path("eval/preds_finetuned.jsonl"), few_shot=False)

rows = [
    ("Zero-Shot Base", zeroshot_report["metrics"]),
    ("Few-Shot Base (3-shot)", fewshot_report["metrics"]),
    ("Fine-Tuned (QLoRA)", ft_report["metrics"]),
]
print("\n| Setup | Valid JSON | Exact Match (All) | Sentiment F1 | Category F1 | Urgency F1 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for name, m in rows:
    print(f"| {name} | {m['valid_schema_rate']:.1%} | {m['all_labels_exact_match']:.1%} | {m['sentiment_macro_f1']:.3f} | {m['category_macro_f1']:.3f} | {m['urgency_macro_f1']:.3f} |")